In [1]:
# Setup: load the shared helpers from lake.py (same folder)
import dlt
from dlt.sources.sql_database import sql_database
from lake import (POSTGRES_URL, DB_TABLES, PIPELINES_DIR,
                  lake_destination, add_metadata, list_lake, read_lake)

print("dlt version:", dlt.__version__)
print("Database:", POSTGRES_URL)

dlt version: 1.31.0
Database: postgresql+psycopg2://pawhome:pawhome@db:5432/pawhome


In [2]:
list_lake("", only_parquet=False)

,key,size_kb,last_modified
0,main.py,2.1,2026-10-29 20:08:25.521+00:00
1,uploads/docs/main.py,2.1,2026-10-29 20:09:28.240+00:00


In [3]:
def build_full_source():
    source = sql_database(POSTGRES_URL, table_names=DB_TABLES)
    for name in DB_TABLES:
        add_metadata(source.resources[name],
                     source_system="postgres",
                     source_resource=name,
                     ingestion_type="full")
    return source

In [4]:
full_pipeline = dlt.pipeline(
    pipeline_name="pawhome_full",
    destination=lake_destination("silo"),
    dataset_name="postgres",
    pipelines_dir=PIPELINES_DIR,
)

info = full_pipeline.run(build_full_source(), write_disposition="replace", loader_file_format="parquet")
print(info)
print("Rows per table:", full_pipeline.last_trace.last_normalize_info.row_counts)

Pipeline pawhome_full load step finished in 0.45 seconds
1 load package(s) were loaded to destination filesystem and into dataset postgres
The filesystem destination used s3://pawhome-lake location to store data
Load package 1793305219.527082 is LOADED and contains no failed jobs
Rows per table: {'adopters': 2000, 'animals': 3000, 'applications': 5000, 'shelters': 50, '_dlt_pipeline_state': 1}


In [5]:
full_keys = list_lake("postgres")
full_keys = full_keys[full_keys.key.str.match(r"postgres/(shelters|adopters|animals|applications)/")]
full_keys

,key,size_kb,last_modified
0,postgres/adopters/ingestion_date=2026-10-29/17...,174.2,2026-10-29 20:20:21.443+00:00
1,postgres/animals/ingestion_date=2026-10-29/179...,113.3,2026-10-29 20:20:21.462+00:00
2,postgres/applications/ingestion_date=2026-10-2...,215.4,2026-10-29 20:20:21.476+00:00
3,postgres/shelters/ingestion_date=2026-10-29/17...,10.0,2026-10-29 20:20:21.444+00:00


In [6]:
animals = read_lake("postgres/animals/")
print(animals.shape[0], "rows")
animals[["id", "name", "species", "status", "_source_system", "_source_resource",
         "_ingestion_type", "_ingested_at", "_load_id"]].head()

3000 rows


,id,name,species,status,_source_system,_source_resource,_ingestion_type,_ingested_at,_load_id
0,729,Oreo,cat,available,postgres,animals,full,2026-10-29 20:20:19.282717+00:00,1793305219.527082
1,790,Gus,dog,available,postgres,animals,full,2026-10-29 20:20:19.282717+00:00,1793305219.527082
2,1349,Coco,cat,available,postgres,animals,full,2026-10-29 20:20:19.282717+00:00,1793305219.527082
3,2075,Rosie,dog,reserved,postgres,animals,full,2026-10-29 20:20:19.282717+00:00,1793305219.527082
4,2319,Buddy,cat,reserved,postgres,animals,full,2026-10-29 20:20:19.282717+00:00,1793305219.527082


In [7]:
before = set(full_keys.key)

info = full_pipeline.run(build_full_source(), write_disposition="replace", loader_file_format="parquet")
print("Rows per table:", full_pipeline.last_trace.last_normalize_info.row_counts)

after_df = list_lake("postgres")
after = set(after_df[after_df.key.str.match(r"postgres/(shelters|adopters|animals|applications)/")].key)

print("\nREMOVED (old run):")
for k in sorted(before - after): print("  -", k)
print("\nADDED (new run):")
for k in sorted(after - before): print("  +", k)

Rows per table: {'adopters': 2000, 'animals': 3000, 'applications': 5000, 'shelters': 50}

REMOVED (old run):
  - postgres/adopters/ingestion_date=2026-10-29/1793305219.527082.11131e94db.parquet
  - postgres/animals/ingestion_date=2026-10-29/1793305219.527082.32e1a615c4.parquet
  - postgres/applications/ingestion_date=2026-10-29/1793305219.527082.f2c550bdcf.parquet
  - postgres/shelters/ingestion_date=2026-10-29/1793305219.527082.ba25be5111.parquet

ADDED (new run):
  + postgres/adopters/ingestion_date=2026-10-29/1793305288.0328362.796db98b81.parquet
  + postgres/animals/ingestion_date=2026-10-29/1793305288.0328362.af8c56d765.parquet
  + postgres/applications/ingestion_date=2026-10-29/1793305288.0328362.5e023ee056.parquet
  + postgres/shelters/ingestion_date=2026-10-29/1793305288.0328362.373b1438a2.parquet


In [8]:
animals = read_lake("postgres/animals/")
print(animals.shape[0], "rows, load ids in the lake:", animals["_load_id"].unique())

3000 rows, load ids in the lake: <ArrowStringArray>
['1793305288.0328362']
Length: 1, dtype: str


**Conclusion:** after the rerun each table still has exactly one file, with a new load id,
and the row count is the same. `replace` throws away the old copy and writes a fresh
complete one. There are never duplicates, but there's no history, and everything is
re-read every time (slow for big tables). That's why Part 3 uses incremental loading.